# Raw 데이터 수집

국가법령정보 API에서 부동산/임대차 관련 문서를 수집하여 `data/raw/{target}.jsonl`에 저장한다.

| target  | 설명                  | 중복 제거 기준       |
| ------- | --------------------- | -------------------- |
| `acr`   | 국민권익위원회 결정문 | `결정문일련번호`     |
| `eflaw` | 현행법령              | `법령ID`             |
| `prec`  | 판례                  | `판례일련번호`       |
| `expc`  | 법령해석례            | `법령해석례일련번호` |

> 각 섹션은 독립적으로 실행 가능. 전체 실행 시 **Kernel → Restart & Run All**.


## 0. 공통 설정


In [1]:
import sys

sys.path.insert(0, "..")
from src.api import fetch_list, save_raw, fetch_details

## 2. 법령해석례(expc)
- 법의 문구가 다소 모호하거나 실제 적용하기 어려운 경우, 행정기관이 일반적/추상적인 법령의 의미를 명확히 하여 내놓은 사례

In [2]:
# 1. 임대차 관련 키워드 수집
EXPC_QUERIES: list[str] = [
    "임차",
    "임대차",
    "부동산임대"
    "전세",
    # "월세",       # 1건 조회되는데 주제와 무관하여 제외
    "임차인",
    "임대인",
    # "전입신고",     # 10건 조회되는데 주제와 무관하여 제외
    # "원상복구",     # 주제와 무관한 내용이 많아 제외
    # "주택임대",     # 주제와 무관한 내용이 많아 제외
    "계약갱신",
    "보증금반환",
    "확정일자",
]

In [3]:
# 2. 수집된 키워드를 본문 내용에 포함하고 있는 법령해석례 추출
seen_expc: set[str] = set()  # 수집된 법령해석례일련번호 추적
expc_items: list[dict] = []

for query in EXPC_QUERIES:
    items = fetch_list("expc", query=query, max_items=None, extra_params={"search": 2})

    # 위 키워드가 하나라도 들어간 법령해석례를 모두 수집한 후 set()로 중복 제거하므로 for문 그대로 이용
    for item in items:
        doc_id = str(item.get("법령해석례일련번호", ""))
        # doc_id가 비어있거나 이미 수집한 항목이면 skip
        if doc_id and doc_id not in seen_expc:
            seen_expc.add(doc_id)
            expc_items.append(item)

save_raw(expc_items, target="expc", mode="w")

print(len(expc_items))

369


In [4]:
# 3. 위 2번에서 추출한 법령해석례들의 내용을 추가
expc_details = fetch_details(
  target="expc",
  items=expc_items,
  id_field="법령해석례일련번호",
)

# 목록만 있는 expc.jsonl 덮어쓰기
result = save_raw(expc_details, target="expc", mode="w")
print(len(expc_details))

369


---

## 5. 수집 결과 요약


In [5]:
from pathlib import Path

# JSONL은 한 줄 = 한 건이므로 줄 수 = 수집 건수
path = Path("../data/raw/expc.jsonl")

if path.exists():
    count = sum(1 for _ in path.open(encoding="utf-8"))
    print(f"expc: {count:5d}건  →  {path}")
else:
    print(f"expc: 파일 없음")

expc:   369건  →  ..\data\raw\expc.jsonl


# 데이터 전처리

In [6]:
import numpy as np
import pandas as pd

### 데이터 로드 및 정제

In [7]:
df = pd.read_json('../data/raw/expc.jsonl', lines=True)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 369 entries, 0 to 368
Data columns (total 11 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   id         369 non-null    int64 
 1   안건명        369 non-null    str   
 2   법령해석례상세링크  369 non-null    str   
 3   회신기관명      369 non-null    str   
 4   안건번호       369 non-null    str   
 5   법령해석례일련번호  369 non-null    int64 
 6   회신기관코드     369 non-null    int64 
 7   질의기관명      369 non-null    str   
 8   회신일자       369 non-null    str   
 9   질의기관코드     369 non-null    str   
 10  본문         369 non-null    object
dtypes: int64(3), object(1), str(7)
memory usage: 31.8+ KB


In [8]:
# id ~ 질의기관코드 컬럼까지 단일 값을 갖는다.
# 본문 컬럼은 'ExpcService' 컬럼 안에 딕셔너리 요소를 가진다.
df.iloc[0]

id                                                           1
안건명          감사원 - 외국회사 소유의 항공기를 직접 사용하기 위하여 운용리스 방식으로 수입하는...
법령해석례상세링크    /DRF/lawService.do?OC=skn24&target=expc&ID=332...
회신기관명                                                      법제처
안건번호                                                   21-0624
법령해석례일련번호                                               332741
회신기관코드                                                 1170000
질의기관명                                                      감사원
회신일자                                                2021.12.07
질의기관코드                                                 1040000
본문           {'ExpcService': {'해석기관코드': '1170000', '안건번호': ...
Name: 0, dtype: object

In [9]:
# '본문' 컬럼의 'ExpcService' 컬럼의 값
df.iloc[0]['본문']['ExpcService']

{'해석기관코드': '1170000',
 '안건번호': '21-0624',
 '이유': '「지방세법」 제6조제1호에서는 “취득”을 매매, 교환, 상속, 증여, 기부, 법인에 대한 현물출자, 건축, 개수(改修), 공유수면의 매립, 간척에 의한 토지의 조성 등과 그 밖에 이와 유사한 취득으로서 원시취득(수용재결로 취득한 경우 등 과세대상이 이미 존재하는 상태에서 취득하는 경우는 제외함), 승계취득 또는 유상ㆍ무상의 모든 취득으로 규정하고 있고, 같은 법 제7조제1항에서는 항공기를 취득한 자에게 취득세를 부과하도록 규정하면서, 같은 조 제2항에서는 관련 법령에 따른 등기ㆍ등록 등을 하지 않은 경우라도 사실상 취득하면 취득한 것으로 보도록 규정하고 있으며, 같은 조 제6항에서는 외국인 소유의 취득세 과세대상 물건을 직접 사용하기 위하여 임차하여 수입하는 경우에는 수입하는 자가 취득한 것으로 보도록 하는 간주취득세 규정을 두고 있습니다.이와 같은 간주취득세 규정은 「대한민국헌법」상의 기본이념인 평등의 원칙을 조세법률관계에 구현한 실천적 원리인 실질과세의 원칙을 반영한 것으로, 과세요건사실의 형식이나 외관에도 불구하고 실질에 따라 담세력이 있는 곳에 과세함으로써 부당한 조세회피행위를 규제하고 과세의 형평을 제고하여 조세정의를 실현하려는 데에 주된 목적이 있는바2)2) 대법원 2012. 1. 19. 선고 2008두8499 전원합의체 판결례 참조, 「지방세법」 제7조제6항에 따른 “임차”의 경우 원칙적으로 과세요건사실의 형식이나 외관이 곧바로 같은 법 제6조제1호에서 의미하는 “취득”에 해당하는 것은 아니지만, “수입”이라는 특수한 상황에서 임차기간 종료 후 수입하는 자의 소유권 취득이 전제되어 실질적으로 취득한 것으로 볼 여지가 있는 임차를 대상으로, 임차 종료 후 실제 취득 시점에서는 잔존가치가 지나치게 낮아져 과세의 형평이나 취득세 관련 규정의 목적에 비추어 취득세 과세표준을 그대로 적용하는 것이 적절하지 않은 경우에는 담세력이 있는 수입하는 자(임차인)에게 취득세를 

**법령해석례 항목별 분류**
1. `Metadata`
- 안건명 : 해석을 요청하는 질의의 제목. 질의기관명과 질의요지가 간략하게 들어감
- 법령해석례상세링크 : 해당 법령해석례의 링크
- 안건번호 : 각 질의의 고유번호. yy-nnnn의 형태로 yy는 질의 요청 연도, nnnn은 해당 연도의 질의 요청 순서를 의미하는 것으로 추정
- 법령해석례일련번호 : 각 법령해석례의 고유 일련번호. '본문' 내용을 가져오기 위해 필요
- 회신일자 : 질의에 대해 법제처가 회신한 날짜. 시간 정보가 있어 Metadata에 저장

2. `Content`
- 질의요지 : 질의 기관이 해석을 요청한 경위에 대해 간략히 표현한 것
- 이유 : 법제처의 해석에 대한 내용

3. `제외`
- id : 고유값이 아니며 중복이 있어 의미가 없음
- 회신기관명 : 100% 법제처 회신이서 Metadata로서의 의미가 없어 제외. `추가 논의 예정`
- 회신기관코드 : 기관의 코드값은 필요 없음
- 질의기관명 : 안건명에 포함된 정보이므로 제외
- 질의기관코드 : 기관의 코드값은 필요 없음
- 해석기관코드 : '회신기관코드'와 동일. 필요 없음
- 해석기관명 : '회신기관명'과 동일. 필요 없음
- 관리기관코드 : 99.9% 결측치 있어서 제외
- 해석일자 : '회신일자'와 동일. 필요 없음
- 등록일시 : 질의에 대한 답변을 등록한 날짜. '회신일자'가 있으므로 제외
- 회답 : 법제처의 해석. '이유' 컬럼이 있으므로 제외

In [10]:
# 항목별 분류에 따라 필요없는 컬럼 제거
df = df[['안건명', '법령해석례상세링크', '안건번호', '법령해석례일련번호', '회신일자', '본문']]
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 369 entries, 0 to 368
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   안건명        369 non-null    str   
 1   법령해석례상세링크  369 non-null    str   
 2   안건번호       369 non-null    str   
 3   법령해석례일련번호  369 non-null    int64 
 4   회신일자       369 non-null    str   
 5   본문         369 non-null    object
dtypes: int64(1), object(1), str(4)
memory usage: 17.4+ KB


In [11]:
# '본문' 컬럼에서 'ExpcService' 키의 값만 추출하여 다시 '본문'에 저장
df['본문'] = df['본문'].apply(lambda x: x['ExpcService'] if isinstance(x, dict) else x)

# '본문' 컬럼 내 '질의요지' 키의 값 추출
df['질의요지'] = df['본문'].apply(lambda x: x['질의요지'] if isinstance(x, dict) else x)
# '본문' 컬럼 내 '이유' 키의 값 추출
df['이유'] = df['본문'].apply(lambda x: x['이유'] if isinstance(x, dict) else x)

# '본문' 컬럼 삭제
df = df[['안건명', '법령해석례상세링크', '안건번호', '법령해석례일련번호', '회신일자', '질의요지', '이유']]

In [12]:
# 컬럼 처리 결과
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 369 entries, 0 to 368
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   안건명        369 non-null    str  
 1   법령해석례상세링크  369 non-null    str  
 2   안건번호       369 non-null    str  
 3   법령해석례일련번호  369 non-null    int64
 4   회신일자       369 non-null    str  
 5   질의요지       369 non-null    str  
 6   이유         369 non-null    str  
dtypes: int64(1), str(6)
memory usage: 20.3 KB


In [13]:
# 중복값 확인
for column in df.columns:
    print(f'{column} : {df[column].duplicated().sum()}개 중복')

안건명 : 57개 중복
법령해석례상세링크 : 0개 중복
안건번호 : 0개 중복
법령해석례일련번호 : 0개 중복
회신일자 : 104개 중복
질의요지 : 58개 중복
이유 : 58개 중복


In [14]:
# 회신일자는 날짜 유형이므로 중복의 여지가 있다고 판단
# Content로 분류한 '질의요지'와 '이유'가 동시에 중복되는 행의 개수 확인
duplicate_count = df.duplicated(['질의요지', '이유']).sum()
print(f"두 컬럼 모두 중복되는 행의 개수: {duplicate_count}")

두 컬럼 모두 중복되는 행의 개수: 58


In [15]:
# 두 컬럼의 내용이 모두 일치할 때 중복 삭제
df = df.drop_duplicates(subset=['질의요지', '이유'], keep='first')
df.info()

<class 'pandas.DataFrame'>
Index: 311 entries, 0 to 368
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   안건명        311 non-null    str  
 1   법령해석례상세링크  311 non-null    str  
 2   안건번호       311 non-null    str  
 3   법령해석례일련번호  311 non-null    int64
 4   회신일자       311 non-null    str  
 5   질의요지       311 non-null    str  
 6   이유         311 non-null    str  
dtypes: int64(1), str(6)
memory usage: 19.4 KB
